In [ ]:
import pandas as pd

# 读取文件
def read_excel_file(file_path):
    """
    此函数用于读取指定路径下的 Excel 文件，并获取指定工作表中的数据
    :param file_path: Excel 文件的路径
    :return: 包含数据的 DataFrame
    """
    # 使用 pandas 的 ExcelFile 函数读取 Excel 文件
    excel_file = pd.ExcelFile(file_path)
    # 获取名为 'Sheet1' 的工作表中的数据
    df = excel_file.parse('Sheet1')
    return df

# 处理缺失值
def handle_missing_values(df):
    """
    此函数用于处理数据中的缺失值，使用均值填充瞬时机械钻速列的缺失值
    :param df: 包含数据的 DataFrame
    :return: 处理缺失值后的 DataFrame
    """
    # 使用均值填充瞬时机械钻速(m/h)列的缺失值
    df['瞬时机械钻速(m/h)'].fillna(df['瞬时机械钻速(m/h)'].mean(), inplace=True)
    return df

# 定义一个函数来判断附近行是否符合裂缝出现的特征
def check_and_correct(row_index, df, window_size=6):
    """
    此函数用于检查并校正数据中裂缝标记的准确性
    :param row_index: 当前行的索引
    :param df: 包含数据的 DataFrame
    :param window_size: 检查的窗口大小，默认为 6
    :return: 如果进行了校正操作返回 True，否则返回 False
    """
    # 确定检查的起始索引，确保不小于 0
    start_index = max(0, row_index - window_size)
    # 确定检查的结束索引，确保不超过数据的长度
    end_index = min(len(df), row_index + window_size + 1)

    for i in range(start_index, end_index):
        if i != row_index:
            # 获取当前行的数据
            current_row = df.iloc[i]
            # 获取目标行的数据
            target_row = df.iloc[row_index]

            # 检查钻压变化，计算当前行和目标行钻压差值的绝对值
            pressure_change = abs(current_row['钻压(kN)'] - target_row['钻压(kN)'])
            # 检查瞬时机械钻速变化，计算当前行和目标行钻速的差值
            speed_change = current_row['瞬时机械钻速(m/h)'] - target_row['瞬时机械钻速(m/h)']
            # 检查立管压力变化，计算目标行和当前行立管压力的差值
            standpipe_pressure_change = target_row['立管压力(MPa)'] - current_row['立管压力(MPa)']

            # 判断是否满足裂缝出现的特征
            if (pressure_change > 1  # 假设钻压变化超过 1 kN 为显著变化
                    and speed_change > 0.5  # 假设钻速增加超过 0.5 m/h 为显著变化
                    and standpipe_pressure_change > 0.3):  # 假设立管压力下降超过 0.3 MPa 为显著变化
                # 如果符合条件，交换是否存在裂缝的值
                df.at[i, '是否存在裂缝'], df.at[row_index, '是否存在裂缝'] = \
                    df.at[row_index, '是否存在裂缝'], df.at[i, '是否存在裂缝']
                return True
    return False

# 遍历数据中裂缝为 '有' 的行进行校准
def calibrate_data(df):
    """
    此函数用于遍历数据中裂缝标记为 '有' 的行，并调用 check_and_correct 函数进行校准
    :param df: 包含数据的 DataFrame
    :return: 校准后的 DataFrame
    """
    # 遍历数据中裂缝为 '有' 的行
    for index, row in df[df['是否存在裂缝'] == '有'].iterrows():
        check_and_correct(index, df)
    return df

# 保存校准后的数据
def save_calibrated_data(df, output_file_path):
    """
    此函数用于将校准后的 DataFrame 保存为 Excel 文件
    :param df: 校准后的 DataFrame
    :param output_file_path: 输出文件的路径
    """
    # 将校准后的数据保存为 Excel 文件，不保存行索引
    df.to_excel(output_file_path, index=False)

# 主函数，整合上述步骤
def main():
    # 定义输入文件路径
    input_file_path = '/mnt/训练数据.xlsx'
    # 读取文件
    df = read_excel_file(input_file_path)
    # 处理缺失值
    df = handle_missing_values(df)
    # 校准数据
    df = calibrate_data(df)
    # 定义输出文件路径
    output_file_path = '/mnt/训练数据_校准后.xlsx'
    # 保存校准后的数据
    save_calibrated_data(df, output_file_path)

if __name__ == "__main__":
    main()